# references with year and source — and the one pass over the dump

The citation edge list with **publication year and source (journal) attached to both endpoints**,
written to `Dimensions/output/references_w_year/`, plus the two per-publication side tables the
rest of the pipeline reads instead of touching the dump again. This is the Dimensions twin of
`OpenAlex/notebook/referenced_works_w_year.ipynb`; the output carries the **same six column names**
so anything written against the OpenAlex edge table reads this one.

| in | |
|---|---|
| `publications/publications_*` | 4,219 extension-less parquet files, 155.5M rows, one `year` per file (unordered) |
| `id` (`pub.…`), `year`, `type`, `document_type`, `source_id`, `journal`, `reference_ids[]`, `citations_count`, `metrics.times_cited`, `categories.for_2020_v2022`, `authors[]`, `doi` | the columns read; `abstract`, `title`, `concepts`, `citations[]` are never touched |

| out | |
|---|---|
| `cache/pub_scalars/part_NNNN.parquet` | one row per publication: year, type, class, source, reference count, FoR codes and names, citation counts, author count, doi |
| `cache/pub_authors/part_NNNN.parquet` | one row per publication with authors: `author_list` (researcher ids in author order), `team_size`, first / last / corresponding author, countries |
| `cache/pub_year_source_map.npz` | one row per publication: sorted `code`, `year` (int16, −1 unknown), `source` (int64 `jour.` code, −1 none) |
| `output/references_w_year/part_NNNN.parquet` | one row per reference edge: `work_id, work_year, work_id_source_id, referenced_work_id, referenced_work_year, referenced_work_id_source_id` |

**Why the map has to be global.** References cross files: a 2004 article cites 1960s papers held in
other files, so attaching years file by file would leave the cited side null. Pass 1 reads the
scalars of every publication, pass 2 maps both endpoints of every edge against that.

**How the map fits in memory.** Both id families are a prefix and digits — `pub.1046637280`,
`jour.1137841` — so `int(id[4:])` / `int(id[5:])` round-trip through int64. The map is three
arrays (~3 GB) and the lookup is `np.searchsorted`.

**Which source.** Dimensions gives each publication one `source_id` (journal, proceedings, book
series, …) and, for journal content, a `journal.id` equal to it. `source_id` is used, `journal.id`
fills the rare gap. Whether the source is a *journal* is decided later from `source_titles.type`.

**Cost.** Pass 1 opens every file once with DuckDB (nested columns, one process per file, 8 in
parallel); pass 2 explodes `reference_ids` with pyarrow. Budget a few hours on a compute node and
~40 GB. **Not a login node.** Finished parts are skipped, so a killed run resumes on resubmission.

Smoke test: `NB_DIM_BASE=<scratch> NB_FILE_LIMIT=12` runs the whole notebook on twelve files into a
scratch folder and touches nothing here.

In [1]:
# (1) Setup
import os, sys, gc, glob, time, json
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import duckdb
from concurrent.futures import ProcessPoolExecutor

sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim

FILES = dim.pub_files()
for d in (dim.SCALARS, dim.AUTHORS, dim.REF_PQ):
    os.makedirs(d, exist_ok=True)
COMPRESSION = 'zstd'
YEAR_MIN, YEAR_MAX = dim.YEAR_MIN, dim.YEAR_MAX   # rejects impossible years only, does not narrow scope

print(f'dump       : {dim.PUBS}')
print(f'files      : {len(FILES)}' + (f'   (NB_FILE_LIMIT={dim.FILE_LIMIT}: strided sample, smoke test)' if dim.FILE_LIMIT else ''))
print(f'scalars    : {dim.SCALARS}')
print(f'authors    : {dim.AUTHORS}')
print(f'map        : {dim.MAP_NPZ}')
print(f'edges      : {dim.REF_PQ}')
print(f'workers    : {dim.N_WORKERS}')

dump       : /project/jevans/dimensions/dimensions/dimensions_june_2025/publications
files      : 4219
scalars    : /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_scalars
authors    : /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_authors
map        : /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
edges      : /project/jevans/Dawoon/Science of Science/Dimensions/output/references_w_year
workers    : 8


## (2) Footer index — what each file holds

Every publications file carries min/max statistics for `year` and `type` per row group, so the
year and type of every file are known without reading a data page (~2.5 min for all 4,219
footers, cached). Nothing is filtered on it here — every file is read — but it is what tells us
the corpus shape up front and lets a later notebook open only the files it needs.

In [2]:
# (2) Footer index (cached). dim.footer_stats is module-level so the pool can pickle it.
t0 = time.time()
idx = pd.read_parquet(dim.INDEX_PQ) if os.path.exists(dim.INDEX_PQ) else None
if idx is None or set(idx['file']) != {os.path.basename(f) for f in FILES}:
    with ProcessPoolExecutor(max_workers=dim.N_WORKERS) as ex:
        idx = pd.DataFrame(list(ex.map(dim.footer_stats, FILES, chunksize=16)))
    idx.to_parquet(dim.INDEX_PQ, index=False)
    print(f'indexed {len(idx):,} footers in {time.time()-t0:.0f}s -> {dim.INDEX_PQ}')
else:
    print(f'footer index loaded: {dim.INDEX_PQ}')
print(f'{len(idx):,} files | {idx.rows.sum():,} publications | {idx.MB.sum()/1e3:.0f} GB | '
      f'single-year files {(idx.year_min == idx.year_max).sum():,} | no statistics {(~idx.stats_ok).sum()}')
by_year = idx.dropna(subset=['year_min']).groupby('year_min')['rows'].sum()
print(f'years {int(by_year.index.min())}-{int(by_year.index.max())}; rows per decade:')
print((by_year.groupby((by_year.index // 10) * 10).sum()).to_string())
print('\nrows by type (files with statistics):')
print(idx[idx.stats_ok].groupby('types')['rows'].sum().sort_values(ascending=False).head(12).to_string())

indexed 4,219 footers in 8s -> /project/jevans/Dawoon/Science of Science/Dimensions/cache/publications_file_index.parquet
4,219 files | 155,507,994 publications | 490 GB | single-year files 4,208 | no statistics 1
years 1665-2025; rows per decade:
year_min
1660.0     1654388
1900.0      611778
1910.0      671694
1920.0      924232
1930.0     1208077
1940.0     1250308
1950.0     2709901
1960.0     4527782
1970.0     6982774
1980.0    10079121
1990.0    14903372
2000.0    24602201
2010.0    45533281
2020.0    39787216

rows by type (files with statistics):
types
article                116385051
chapter                 10938020
proceeding               7500112
article|proceeding       3614329
preprint                 3309322
article|chapter          3257943
article|monograph        2152719
chapter|monograph        1714831
preprint|proceeding      1390539
chapter|proceeding       1366489
monograph|preprint        794458
article|book              724743


## (3) Pass 1 — scalars and authors, one part per file

One DuckDB query per file, its own process, one thread. The nested columns are unnested here
and never again: FoR division and group codes and names, the resolved author ids in author order,
the distinct author countries. Two parts per file, written to `.tmp` and renamed only after the
footer is down; an existing readable part is skipped.

Column notes, `pub_scalars`:

| column | from |
|---|---|
| `pid`, `pub_id` | `int(id[4:])`, `id` |
| `year`, `type`, `doc_class`, `is_citable` | `year`, `type`, `document_type.classification`, `document_type.is_citable` |
| `source_id`, `journal_id`, `journal_title` | `source_id`, `journal.id`, `journal.title` |
| `ref_count` | `len(reference_ids)` — references within the dump, the analogue of the OpenAlex edge count |
| `citations_count`, `times_cited` | Dimensions' own forward-citation counts (`citations_count`, `metrics.times_cited`); carried for comparison with the graph-derived `C_all` |
| `for1_codes`, `for1_names`, `for2_codes` | `categories.for_2020_v2022` first level (2-digit divisions) and second level (4-digit groups), `;`-joined in the dump's order |
| `n_authors` | `len(authors)` — every author slot, resolved or not |

`pub_authors` (publications with at least one author slot): `author_list` = researcher ids of the
**resolved** slots in author order (an unresolved author has no id and cannot be listed);
`team_size` = all slots; `first_author` / `last_author` = researcher id of the first / last slot
(null when that slot is unresolved); `countries` = distinct ISO2 codes over every author's
`affiliations_address`, sorted; `corresponding_author` = the first slot flagged corresponding.

In [3]:
# (3) Pass 1 -- the worker is dim.scalars_part (DuckDB, one thread, one file); the SQL that
# unnests the row lives there so the same extraction is importable from any notebook.
import inspect
print(inspect.getsource(dim.scalars_part).split('con.execute(f"""')[1].split('""")')[0].strip()[:1500])

t0 = time.time(); total = skipped = 0
with ProcessPoolExecutor(max_workers=dim.N_WORKERS) as ex:
    for k, (i, n, was_done) in enumerate(ex.map(dim.scalars_part, list(enumerate(FILES)), chunksize=1)):
        total += n; skipped += int(was_done)
        if (k + 1) % 100 == 0 or k + 1 == len(FILES):
            el = time.time() - t0
            print(f'  {k+1}/{len(FILES)} files  {total:,} rows read  [{el:.0f}s, '
                  f'~{el/(k+1)*(len(FILES)-k-1)/60:.0f}m left]', flush=True)
print(f'\n[{time.time()-t0:.0f}s] pass 1 done: {len(dim.scalar_parts())} scalar parts, '
      f'{len(dim.author_parts())} author parts ({skipped} were already there)')

CREATE TEMP TABLE x AS
    SELECT id AS pub_id, TRY_CAST(substr(id, 5) AS BIGINT) AS pid,
           TRY_CAST(year AS INTEGER) AS year, type,
           document_type.classification AS doc_class, document_type.is_citable AS is_citable,
           source_id, journal.id AS journal_id, journal.title AS journal_title,
           coalesce(len(reference_ids), 0) AS ref_count,
           citations_count, metrics.times_cited AS times_cited,
           array_to_string(list_transform(categories.for_2020_v2022.first_level.full,  f -> f.code), ';') AS for1_codes,
           array_to_string(list_transform(categories.for_2020_v2022.first_level.full,  f -> f.name), ';') AS for1_names,
           array_to_string(list_transform(categories.for_2020_v2022.second_level.full, f -> f.code), ';') AS for2_codes,
           coalesce(len(authors), 0) AS n_authors, doi,
           authors
    FROM read_parquet('{src}')
  100/4219 files  3,830,471 rows read  [12s, ~8m left]
  200/4219 files  6,740,148 rows read  

## (4) The map — code, year, source for every publication

From the scalar parts: sorted int64 code, int16 year (−1 outside `YEAR_MIN..YEAR_MAX` or null),
int64 source code (`source_id`, else `journal.id`, else −1). `pid` must be unique across files;
a duplicate would mean one publication in two files and is a hard error, not a de-duplication.

In [4]:
# (4) Build the map
def build_map(force=False):
    if os.path.exists(dim.MAP_NPZ) and not force:
        z = np.load(dim.MAP_NPZ)
        print(f'map loaded: {len(z["code"]):,} publications')
        return z['code'], z['year'], z['source']
    t0 = time.time()
    con = duckdb.connect(); con.execute("SET memory_limit='60GB'"); con.execute('SET preserve_insertion_order=false')
    d = con.execute(f"""
        SELECT pid, year, coalesce(source_id, journal_id) AS src
        FROM read_parquet('{dim.SCALARS}/part_*.parquet') WHERE pid IS NOT NULL ORDER BY pid""").fetch_arrow_table()
    con.close()
    code_ = d.column('pid').to_numpy()
    y = d.column('year').to_numpy(zero_copy_only=False).astype(np.float64)
    y = np.where(np.isfinite(y) & (y >= YEAR_MIN) & (y <= YEAR_MAX), y, -1).astype(np.int16)
    source_ = dim.src_to_code(d.column('src').to_pandas())
    del d; gc.collect()
    dup = int((code_[1:] == code_[:-1]).sum())
    assert dup == 0, f'{dup:,} publication ids appear in more than one file'
    np.savez(dim.MAP_NPZ, code=code_, year=y, source=source_)
    print(f'[{time.time()-t0:.0f}s] map: {len(code_):,} publications, '
          f'{(y >= 0).mean()*100:.1f}% with a year, {(source_ >= 0).mean()*100:.1f}% with a source '
          f'-> {dim.MAP_NPZ} ({os.path.getsize(dim.MAP_NPZ)/1e9:.2f} GB)')
    return code_, y, source_


CODE, YEAR, SOURCE = build_map()
print(f'memory: {(CODE.nbytes + YEAR.nbytes + SOURCE.nbytes)/1e9:.2f} GB')

[183s] map: 155,507,994 publications, 100.0% with a year, 85.2% with a source -> /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz (2.80 GB)
memory: 2.80 GB


## (5) Pass 2 — explode `reference_ids`, attach both years and both sources

One output part per input file. `reference_ids` is flattened with pyarrow (`list_flatten` +
`list_parent_indices`, so the citing id rides with each cited id); both sides go through the
map. The worker processes are forked after the map is built, so the three arrays are shared,
not copied. Output written to `.tmp` and renamed after the footer; readable parts are skipped.

In [5]:
# (5) Map both endpoints and write. The map is handed to dim_common BEFORE the pool forks,
# so every worker shares the three arrays; dim.edges_part / dim.lookup do the work.
dim.set_map(CODE, YEAR, SOURCE)

t0 = time.time(); total = skipped = 0
with ProcessPoolExecutor(max_workers=dim.N_WORKERS) as ex:
    for k, (i, n, was_done) in enumerate(ex.map(dim.edges_part, list(enumerate(FILES)), chunksize=2)):
        total += n; skipped += int(was_done)
        if (k + 1) % 100 == 0 or k + 1 == len(FILES):
            el = time.time() - t0
            print(f'  {k+1}/{len(FILES)}  {total:,} edges  [{el:.0f}s, ~{el/(k+1)*(len(FILES)-k-1)/60:.0f}m left]', flush=True)
print(f'\n[{time.time()-t0:.0f}s] {total:,} edges written, {skipped} parts already done')
print(f'-> {dim.REF_PQ}  ({sum(os.path.getsize(p) for p in dim.ref_parts())/1e9:.1f} GB)')

  100/4219  41,394,425 edges  [42s, ~29m left]
  200/4219  99,588,106 edges  [83s, ~28m left]
  300/4219  145,670,487 edges  [118s, ~26m left]
  400/4219  216,786,275 edges  [175s, ~28m left]
  500/4219  279,272,804 edges  [223s, ~28m left]
  600/4219  325,862,941 edges  [262s, ~26m left]
  700/4219  365,769,771 edges  [295s, ~25m left]
  800/4219  424,976,604 edges  [338s, ~24m left]
  900/4219  471,353,739 edges  [379s, ~23m left]
  1000/4219  543,711,163 edges  [434s, ~23m left]
  1100/4219  606,190,829 edges  [481s, ~23m left]
  1200/4219  646,880,319 edges  [517s, ~22m left]
  1300/4219  703,454,537 edges  [558s, ~21m left]
  1400/4219  750,365,181 edges  [598s, ~20m left]
  1500/4219  820,268,413 edges  [654s, ~20m left]
  1600/4219  877,050,915 edges  [696s, ~19m left]
  1700/4219  911,499,283 edges  [724s, ~18m left]
  1800/4219  964,149,429 edges  [768s, ~17m left]
  1900/4219  1,006,673,020 edges  [803s, ~16m left]
  2000/4219  1,060,176,768 edges  [845s, ~16m left]
  2100/42

## (6) Verify

Footers for the counts (the edge total must equal the sum of `ref_count` over the scalar parts);
a sample of parts for the null rates; a spot check that the year attached to a citing id is the
year its own scalar row carries.

In [6]:
# (6) Verify
outs = dim.ref_parts()
n_out = sum(pq.ParquetFile(p).metadata.num_rows for p in outs)
con = duckdb.connect(); con.execute("SET memory_limit='40GB'")
n_ref = con.execute(f"SELECT sum(ref_count) FROM read_parquet('{dim.SCALARS}/part_*.parquet')").fetchone()[0]
n_pub = con.execute(f"SELECT count(*) FROM read_parquet('{dim.SCALARS}/part_*.parquet')").fetchone()[0]
print(f'parts      : {len(outs)}/{len(FILES)}')
print(f'edges      : {n_out:,} written vs {int(n_ref):,} reference slots in the scalars   '
      f"{'OK' if n_out == n_ref else 'MISMATCH'}")
print(f'pubs       : {n_pub:,} in the scalar parts, {len(CODE):,} in the map')

pick = outs[:: max(1, len(outs) // 8)][:8]
d = pd.concat([pq.read_table(p).to_pandas() for p in pick], ignore_index=True)
print(f'\nsampled {len(d):,} edges from {len(pick)} parts')
for c in ('work_year', 'work_id_source_id', 'referenced_work_year', 'referenced_work_id_source_id'):
    print(f'  {c:<32} null {d[c].isna().sum():>10,}  ({d[c].isna().mean()*100:5.2f}%)')
both = d.dropna(subset=['work_year', 'referenced_work_year'])
back = both.work_year - both.referenced_work_year
print(f'\n  citing year - cited year: median {back.median():.0f}, '
      f'{(back < 0).mean()*100:.2f}% negative (citing something published later)')
print(f'  self-citation by source: {(d.work_id_source_id == d.referenced_work_id_source_id).mean()*100:.2f}% of edges stay inside one source')
print(f'  cited id not in the dump: {(d.referenced_work_year.isna() & d.referenced_work_id_source_id.isna()).mean()*100:.2f}% of edges')

# spot check: the citing side's year must equal the scalar row's year
sc = con.execute(f"SELECT pub_id, year FROM read_parquet('{dim.SCALARS}/{dim.part_name(0)}')").df()
ymap = dict(zip(sc.pub_id, sc.year))
s = d[d.work_id.isin(ymap)].head(200_000)
bad = int((s.work_year.astype(float) != s.work_id.map(ymap).astype(float)).sum())
print(f"\n  spot check citing year vs scalars: {len(s):,} edges, {bad} disagreements {'OK' if bad == 0 else 'FAIL'}")
con.close()
display(d.head(8))

parts      : 4219/4219
edges      : 2,142,237,844 written vs 2,142,237,844 reference slots in the scalars   OK
pubs       : 155,507,994 in the scalar parts, 155,507,994 in the map

sampled 3,299,603 edges from 8 parts
  work_year                        null          0  ( 0.00%)
  work_id_source_id                null    261,676  ( 7.93%)
  referenced_work_year             null        764  ( 0.02%)
  referenced_work_id_source_id     null    198,278  ( 6.01%)

  citing year - cited year: median 7, 0.22% negative (citing something published later)
  self-citation by source: 3.88% of edges stay inside one source
  cited id not in the dump: 0.02% of edges

  spot check citing year vs scalars: 200,000 edges, 0 disagreements OK


,work_id,work_year,work_id_source_id,referenced_work_id,referenced_work_year,referenced_work_id_source_id
0,pub.1001190302,1997,jour.1104843,pub.1007118321,1984.0,jour.1017013
1,pub.1001190302,1997,jour.1104843,pub.1014109080,1991.0,jour.1014535
2,pub.1001190302,1997,jour.1104843,pub.1016804550,1992.0,jour.1018982
3,pub.1001190302,1997,jour.1104843,pub.1037338316,1968.0,jour.1017013
4,pub.1001190302,1997,jour.1104843,pub.1051938012,1985.0,jour.1091447
5,pub.1001190302,1997,jour.1104843,pub.1053992084,1995.0,jour.1017745
6,pub.1001190302,1997,jour.1104843,pub.1059470262,1993.0,jour.1089721
7,pub.1001190302,1997,jour.1104843,pub.1059472260,1992.0,jour.1089721


## (7) What this is for

Each edge now carries, for both ends, when it was published and where — the backward citation
age (`work_year − referenced_work_year`) that the C3/C5/C10 windows filter on and the
sleeping-beauty histogram bins, and the **source pair** of a reference, which is the unit the Uzzi
null shuffles. Join `source_titles` on either source column for the title and type;
`dim_common.read_sources()` returns it as `source_id, journal, source_type, issns`.

In [7]:
# (7) A worked look — citation age and most-cited sources
d2 = d.dropna(subset=['work_year', 'referenced_work_year']).copy()
d2['age'] = d2.work_year - d2.referenced_work_year
d2 = d2[d2.work_year.between(1950, 2025) & d2.age.between(0, 100)]
g = d2.groupby((d2.work_year // 10 * 10).astype(int))['age'].agg(['size', 'median', 'mean'])
print('backward citation age by decade of the citing publication (sampled parts):\n')
print(g.rename(columns={'size': 'edges'}).round(1).to_string())

src = dim.read_sources().drop_duplicates('source_id').set_index('source_id')['journal']
top = d.dropna(subset=['referenced_work_id_source_id']).referenced_work_id_source_id.value_counts().head(10)
print('\nmost-cited sources in the sample:')
for sid, cnt in top.items():
    print(f'  {sid:<16}{cnt:>9,}  {src.get(sid, "(not in source_titles)")}')

backward citation age by decade of the citing publication (sampled parts):

             edges  median  mean
work_year                       
1960         33464     5.0   8.1
1990        374510     6.0   9.1
2000        303129     5.0   7.4
2010        659630     8.0  10.8
2020       1918362     7.0   9.8

most-cited sources in the sample:
  jour.1018957       37,875  Nature
  jour.1082971       32,317  Proceedings of the National Academy of Sciences of the United States of America
  jour.1346339       30,175  Science
  jour.1037553       23,279  PLOS ONE
  jour.1077138       18,494  Journal of Biological Chemistry
  jour.1222150       17,507  Lecture Notes in Computer Science
  jour.1043282       16,032  Nature Communications
  jour.1019114       15,985  Cell
  jour.1045337       15,773  Scientific Reports
  jour.1018277       15,296  Physical Review Letters
